# Lycées en Bretagne
Source : https://data.bretagne.bzh/explore/assets/lycees-en-bretagne/export/

https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/exports/csv/?delimiter=%3B&lang=fr&timezone=Europe%2FParis&use_labels=true

Le fichier csv (Comma-separated values) a été mis dans une base de données(bdd) sqlite3 avec DB Browser for SQLite Version 3.13.1, la bdd s'appelle 'lycées_Bretagne.db'

In [7]:
import sqlite3, pandas, duckdb,glob,os, sys,requests,json, folium
pandas.options.display.max_rows=1000
pandas.set_option('display.max_colwidth', None)
# Create your connection.
LB = sqlite3.connect('lycées_Bretagne.db')
lb=pandas.read_sql_query('SELECT * FROM "lycees-en-bretagne" ;', LB)
lb.columns

Index(['Geo Point', 'Geo Shape', 'gml_id', 'code_rne', 'search_id', 'nom',
       'adresse', 'code_postal', 'ville', 'cedex', 'reseau_lib', 'secteur_li',
       'type_etabl', 'photo', 'x', 'y', 'latitude', 'longitude',
       'code_rne_rectorat', 'commentaire'],
      dtype='str')

In [37]:
lpos=lb[['nom','Geo Point', 'secteur_li','photo']]
lpos['coord']=[(float(elt.split(',')[0]),float(elt.split(',')[1])) for elt in lpos['Geo Point']] #str ->float
lpos['couleur']=["red" if elt  == 'Public' else "blue" for elt in lpos.secteur_li] # lycées pulics en rouge, privés en bleu
lpos

,nom,Geo Point,secteur_li,photo,coord,couleur
0,Lycée Victor et Hélène Basch,"48.125631211490266, -1.6907552264262897",Public,https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/files/5e8f08562187f3e66d3682da5b134c9e,"(48.125631211490266, -1.6907552264262897)",red
1,Lycée Joliot Curie,"48.125996924520614, -1.6484283775092052",Public,https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/files/06e572f3323c1bffc9a44b367dbedb5f,"(48.125996924520614, -1.6484283775092052)",red
2,Lycée Jean-Baptiste Le Taillandier,"48.35720000000286, -1.203599999994624",Privé sous contrat avec l'éducation nationale,NaN,"(48.35720000000286, -1.203599999994624)",blue
3,Lycée Colbert,"47.746000000024274, -3.3819399999978574",Public,https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/files/1175b9a8fa45ee60e3490ed97bfe760c,"(47.746000000024274, -3.3819399999978574)",red
4,Lycée agricole Jean Baptiste Le Taillandier site Edmond Michelet,"48.3628000000044, -1.2044299999965922",Privé sous contrat avec l'éducation nationale,NaN,"(48.3628000000044, -1.2044299999965922)",blue
5,MFREO,"48.04030000002233, -2.409719999999123",Privé sous contrat avec l'éducation nationale,NaN,"(48.04030000002233, -2.409719999999123)",blue
6,Lycée Brocéliande,"47.95060000002077, -2.137380000000905",Public,https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/files/3e3a61d7a27f57882f4a5df2eadb5c3d,"(47.95060000002077, -2.137380000000905)",red
7,LP André-Marie Ampère,"47.95920000001815, -2.5469800000014495",Public,https://data.bretagne.bzh/api/explore/v2.1/catalog/datasets/lycees-en-bretagne/files/87552f985216b948879f3ba8482ae3ca,"(47.95920000001815, -2.5469800000014495)",red
8,Lycée Les Cordeliers,"48.45420000000853, -2.043319999997701",Privé sous contrat avec l'éducation nationale,NaN,"(48.45420000000853, -2.043319999997701)",blue
9,Lycée Agricole Le Nivot,"48.28000000001689, -4.008600000000622",Privé sous contrat avec l'éducation nationale,NaN,"(48.28000000001689, -4.008600000000622)",blue


In [34]:
from folium.plugins import MarkerCluster
from folium.plugins import MeasureControl

m = folium.Map(location=[48.217, -1.89], zoom_start=8)
#m.add_child(MeasureControl())
marker_cluster = MarkerCluster().add_to(m)
for i in range(len(lpos)):
    folium.Marker(
        location=lpos.iloc[i].coord,
        popup=lpos.iloc[i].nom,
        icon=folium.Icon(color=lpos.iloc[i].couleur,icon="school", prefix='fa') #red or blue
    ).add_to(marker_cluster)
m.save(os.path.join('Lycées_Bretagne.html'))
m

## Attention : le fichier 'Lycées_Bretagne.html' ne fonctionne pas en local
Il faut le mettre dans un serveur.

**Exemple simple" : lancer dans le dossier contenant 'Lycées_Bretagne.html'

La commande : python3 -m http.server 1234

Cela lance un serveur web python

Dans Firefox, ouvrez : http://localhost:1234/Lyc%C3%A9es_Bretagne.html


Voici les lycées publiques dans https://quellycee.fr/bretagne/lycees/

In [54]:
# Les lycées Publiques
m = folium.Map(location=[48.217, -1.89], zoom_start=8)
#m.add_child(MeasureControl())
marker_cluster = MarkerCluster().add_to(m)
for i in range(len(lpos)):
    if lpos.iloc[i].couleur == "red" :
        folium.Marker(
            location=lpos.iloc[i].coord,
            popup= lpos.iloc[i].nom, 
            icon=folium.Icon(color=lpos.iloc[i].couleur,icon="school", prefix='fa') #red or blue
        ).add_to(marker_cluster)
m.save(os.path.join('Lycées_publiques_Bretagne.html'))
m

In [36]:
# Les MFR Maison familiale rurale
m = folium.Map(location=[48.217, -1.89], zoom_start=8)
#m.add_child(MeasureControl())
marker_cluster = MarkerCluster().add_to(m)
for i in range(len(lpos)):
    if  "MFR" in lpos.iloc[i].nom :
        folium.Marker(
            location=lpos.iloc[i].coord,
            popup=lpos.iloc[i].nom,
            icon=folium.Icon(color=lpos.iloc[i].couleur,icon="school", prefix='fa') #red or blue
        ).add_to(marker_cluster)
m.save(os.path.join('MFR_Bretagne.html'))
m

In [55]:
# Les lycées Publiques avec les photos
m = folium.Map(location=[48.217, -1.89], zoom_start=8)
#m.add_child(MeasureControl())
marker_cluster = MarkerCluster().add_to(m)
for i in range(len(lpos)):
    if lpos.iloc[i].couleur == "red" :
        foto=lpos.iloc[i].photo
        html = '<img src="{}">'.format
        iframe = IFrame(html(foto), width=1000, height=650)
        folium.Marker(
            location=lpos.iloc[i].coord,
            popup=folium.Popup(iframe, max_width=500),
            icon=folium.Icon(color=lpos.iloc[i].couleur,icon="school", prefix='fa') #red or blue
        ).add_to(marker_cluster)
m.save(os.path.join('Lycées_publiques_Bretagne.html'))
m